In [1]:
#used for face detection and cropping from our dataset images 
%pip install retina-face opencv-python tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
# importing libraries
import os
import cv2
from tqdm import tqdm
from retinaface import RetinaFace

In [3]:
# root path
ROOT = "dataset_split"
SPLITS = ["train", "test"]
SUBSETS = ["gallery", "probe"]

In [4]:
# creating output folders
for split in SPLITS:
    os.makedirs(os.path.join(ROOT, split, "gallery_cropped"), exist_ok=True)
    os.makedirs(os.path.join(ROOT, split, "probe_cropped"), exist_ok=True)

print("Output folders ready.")

Output folders ready.


In [5]:
#cropping resizing and getting the face detection score
def crop_and_resize_face(image_path):

    try:
        img = cv2.imread(image_path)

        if img is None:
            return None, None

        faces = RetinaFace.detect_faces(image_path)

        if not isinstance(faces, dict):
            return None, None

        largest_area = -1
        best_box = None
        best_score = None

        for face_id in faces:

            x1, y1, x2, y2 = faces[face_id]["facial_area"]

            score = faces[face_id]["score"]

            area = (x2 - x1) * (y2 - y1)

            if area > largest_area:

                largest_area = area

                best_box = (x1, y1, x2, y2)

                best_score = score

        if best_box is None:
            return None, None

        x1, y1, x2, y2 = best_box

        h, w = img.shape[:2]

        x1 = max(0, x1)
        y1 = max(0, y1)

        x2 = min(w, x2)
        y2 = min(h, y2)

        face = img[y1:y2, x1:x2]

        face = cv2.resize(face, (112, 112))

        return face, float(best_score)

    except Exception as e:

        print(f"ERROR: {image_path}")

        print(e)

        return None, None

In [6]:
import pandas as pd
metadata_rows=[]

In [7]:
# processing and saving all images
for split in SPLITS:

    for subset in SUBSETS:

        input_root = os.path.join(ROOT, split, subset)
        output_root = os.path.join(ROOT, split, subset + "_cropped")
        identities = os.listdir(input_root)
        print(f"\nProcessing {split}/{subset}")

        for identity in tqdm(identities):
            src_identity_dir = os.path.join(input_root, identity)
            dst_identity_dir = os.path.join(output_root, identity)
            os.makedirs(dst_identity_dir, exist_ok=True)
            images = os.listdir(src_identity_dir)

            for image_name in images:
                image_path = os.path.join(src_identity_dir, image_name)
                cropped_face, det_score = crop_and_resize_face(image_path)

                if cropped_face is None:
                    continue

                save_path = os.path.join(dst_identity_dir, image_name)
                cv2.imwrite(save_path, cropped_face)
                metadata_rows.append(
                    {
                        "split": split,
                        "subset": subset,
                        "identity": identity,
                        "image_name": image_name,
                        "image_path": os.path.abspath(save_path),
                        "det_score": det_score,
                    }
                )
                print(f"{image_name} | Detection Score = {det_score:.4f}")

print("Cropping completed.")
metadata_df = pd.DataFrame(metadata_rows)

metadata_df.to_csv(
    "face_detection_scores.csv",
    index=False
)

print("\nDetection scores saved")

print(metadata_df.head())
print("\nDetection Score Statistics")
print(metadata_df["det_score"].describe())


Processing train/gallery


  0%|          | 0/100 [00:00<?, ?it/s]

Salma_Hayek_0012.jpg | Detection Score = 0.9976
Salma_Hayek_0006.jpg | Detection Score = 0.9998
Salma_Hayek_0013.jpg | Detection Score = 0.9994
Salma_Hayek_0011.jpg | Detection Score = 0.9993
Salma_Hayek_0010.jpg | Detection Score = 0.9995
Salma_Hayek_0004.jpg | Detection Score = 0.9995
Salma_Hayek_0001.jpg | Detection Score = 0.9995
Salma_Hayek_0003.jpg | Detection Score = 0.9990
Salma_Hayek_0002.jpg | Detection Score = 0.9993


  1%|          | 1/100 [00:20<33:50, 20.51s/it]

Salma_Hayek_0008.jpg | Detection Score = 0.9994
Queen_Elizabeth_II_0009.jpg | Detection Score = 0.9993
Queen_Elizabeth_II_0008.jpg | Detection Score = 0.9985
Queen_Elizabeth_II_0006.jpg | Detection Score = 0.9984
Queen_Elizabeth_II_0007.jpg | Detection Score = 0.9991
Queen_Elizabeth_II_0013.jpg | Detection Score = 0.9989
Queen_Elizabeth_II_0005.jpg | Detection Score = 0.9990
Queen_Elizabeth_II_0010.jpg | Detection Score = 0.9989
Queen_Elizabeth_II_0001.jpg | Detection Score = 0.9992
Queen_Elizabeth_II_0003.jpg | Detection Score = 0.9994


...

Megawati_Sukarnoputri_0023.jpg | Detection Score = 0.9993
Nancy_Pelosi_0003.jpg | Detection Score = 0.9986
Nancy_Pelosi_0014.jpg | Detection Score = 0.9995


 97%|█████████▋| 29/30 [04:01<00:07,  7.92s/it]

Nancy_Pelosi_0009.jpg | Detection Score = 0.9987
Pervez_Musharraf_0009.jpg | Detection Score = 0.9990
Pervez_Musharraf_0012.jpg | Detection Score = 0.9990
Pervez_Musharraf_0003.jpg | Detection Score = 0.9995


100%|██████████| 30/30 [04:08<00:00,  8.28s/it]

Pervez_Musharraf_0017.jpg | Detection Score = 0.9991
Cropping completed.

Detection scores saved
   split   subset     identity            image_name  \
0  train  gallery  Salma_Hayek  Salma_Hayek_0012.jpg   
1  train  gallery  Salma_Hayek  Salma_Hayek_0006.jpg   
2  train  gallery  Salma_Hayek  Salma_Hayek_0013.jpg   
3  train  gallery  Salma_Hayek  Salma_Hayek_0011.jpg   
4  train  gallery  Salma_Hayek  Salma_Hayek_0010.jpg   

                                          image_path  det_score  
0  /Users/admin/Desktop/reliable_rejection_under_...   0.997556  
1  /Users/admin/Desktop/reliable_rejection_under_...   0.999787  
2  /Users/admin/Desktop/reliable_rejection_under_...   0.999385  
3  /Users/admin/Desktop/reliable_rejection_under_...   0.999275  
4  /Users/admin/Desktop/reliable_rejection_under_...   0.999498  

Detection Score Statistics
count    3168.000000
mean        0.999085
std         0.000944
min         0.969271
25%         0.998949
50%         0.999218
75%         0.99

In [8]:
# verification
for split in SPLITS:

    for subset in ["gallery_cropped", "probe_cropped"]:
        total = 0
        root = os.path.join(ROOT, split, subset)

        for identity in os.listdir(root):
            total += len(os.listdir(os.path.join(root, identity)))
            
        print(split, subset, total)

train gallery_cropped 2002
train probe_cropped 549
test gallery_cropped 482
test probe_cropped 135
